# BTXRD Anatomy-Region Data Check

Small, standalone notebook: verifies the `upper limb` / `lower limb` / `pelvis` anatomy-region columns in
`dataset.csv` are clean enough to use for anatomy-matched contrastive learning, BEFORE any training code is
built on top of them.

Why this matters: the specific-bone columns (`tibia`, `femur`, `humerus`, `hand`, `ulna`, `radius`, `foot`,
`fibula`, `hip bone`, and the `*-joint` columns) are only ever set to 1 for tumor images in this dataset -- using
them as a classifier input or auxiliary label would leak the tumor label itself ("has a tibia annotation" =>
tumor). The three region columns checked here do NOT have this problem (both tumor and normal images get a
region label), which is what makes them safe to use for region-matched pairing. This notebook confirms that
assumption against the real `dataset.csv` rather than trusting it blindly.

Runs on branch `pipeline-anatomy-contrastive` (not `pipeline` -- this branch adds `anatomy_region` parsing to
`datasets/btxrd.py` and the verification script, does not touch the WSSS pipeline itself, and is safe to run
independently of any other notebook/session).


In [ ]:
# Cell 1 -- paths and repo checkout
from pathlib import Path
import os
import subprocess
import sys

NOTEBOOK_ROOT = Path.cwd()
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

REPO_URL = os.environ.get("BTXRD_REPO_URL", "https://github.com/itsthang333/Thesis.git")
GIT_BRANCH = os.environ.get("BTXRD_GIT_BRANCH", "pipeline-anatomy-contrastive")
DATASET_OVERRIDE = os.environ.get("BTXRD_ROOT", "")

if (NOTEBOOK_ROOT / "project").exists():
    PROJECT_PARENT = NOTEBOOK_ROOT
else:
    PROJECT_PARENT = KAGGLE_WORKING / "Thesis"
PROJECT_DIR = PROJECT_PARENT / "project"

if not PROJECT_DIR.exists():
    PROJECT_PARENT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(PROJECT_PARENT)], check=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("GIT_BRANCH:", GIT_BRANCH)


In [ ]:
# Cell 2 -- locate the BTXRD dataset root
from datasets.btxrd import resolve_btxrd_root

def find_btxrd_root(base):
    candidates = [base, *base.glob("*"), *base.glob("*/*")] if base and base.exists() else []
    for candidate in candidates:
        try:
            return resolve_btxrd_root(candidate)
        except FileNotFoundError:
            pass
    return None

search_root = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else KAGGLE_INPUT
BTXRD_ROOT = find_btxrd_root(search_root)
if BTXRD_ROOT is None:
    raise FileNotFoundError(
        f"BTXRD not found under {search_root}. Set BTXRD_ROOT env var or attach the dataset."
    )
print("BTXRD_ROOT:", BTXRD_ROOT)


## Run the verification script

`tools/check_anatomy_region_labels.py` loads every record via `load_btxrd_records()` (the same loader the
rest of the pipeline uses) and reports:
1. How many images have exactly one region column set (should be ~100%).
2. Per-region tumor/normal counts, to confirm every region has both groups represented (this is exactly what
   makes region-matched pairing possible) and to cross-check against the numbers already sanity-checked by
   hand (upper limb 672 normal / 452 tumor, lower limb 1095 normal / 1311 tumor, pelvis 112 normal / 104 tumor).


In [ ]:
# Cell 3 -- run the check
import subprocess, sys

result = subprocess.run(
    [sys.executable, "tools/check_anatomy_region_labels.py", "--ram-root", str(BTXRD_ROOT)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"check_anatomy_region_labels.py exited with code {result.returncode}")


## Manual cross-check against the classification dataset loader

Confirms `BTXRDClassificationDataset`/`load_btxrd_records()` records agree with the raw CSV read above (no
row dropped/duplicated by the split logic), and prints a quick tumor_type x anatomy_region crosstab so any
skew (e.g. a tumor_type that only ever appears in one region) is visible before designing the region-matched
batch sampler.


In [ ]:
# Cell 4 -- tumor_type x anatomy_region crosstab
import pandas as pd
from datasets.btxrd import ANATOMY_REGION_COLUMNS, TUMOR_TYPE_CLASS_NAMES, load_btxrd_records

records = load_btxrd_records(BTXRD_ROOT)
df = pd.DataFrame(records)
df["tumor_type_name"] = df["tumor_type"].map(dict(enumerate(TUMOR_TYPE_CLASS_NAMES)))
df["anatomy_region_name"] = df["anatomy_region"].map(
    {i: name for i, name in enumerate(ANATOMY_REGION_COLUMNS)}
).fillna("unknown")

print(f"Total records: {len(df)}")
print(f"Records with unknown region: {(df['anatomy_region'] == -1).sum()}")
print()
crosstab = pd.crosstab(df["tumor_type_name"], df["anatomy_region_name"])
display(crosstab)
